# ADS Bibliometric Analysis Pipeline

This notebook is the single entry point for the `ads_bib` package.
It executes all steps sequentially, with configuration cells between steps
so that downstream parameters can be set based on upstream results.

**Pipeline Steps:**
1. Search & Export â€” Query ADS API, resolve bibcodes to metadata
2. Translation â€” Detect languages, translate non-English titles/abstracts
3. Tokenization â€” Create full_text, tokenize with spaCy
4. AND â€” Author Name Disambiguation (placeholder)
5. Topic Modeling & Curation â€” BERTopic + datamapplot + cluster removal
6. Citation Networks â€” Direct, Co-Citation, Bibliographic Coupling, Author Co-Citation

---
## Setup

In [1]:
import os
from pathlib import Path

# Initialize paths (relative to this notebook's location)
from ads_bib.config import init_paths, load_env
from ads_bib.run_manager import RunManager
from ads_bib._utils.costs import CostTracker

paths = init_paths()  # uses CWD = notebook directory
load_env()

# Cost tracker for all API calls (OpenRouter, etc.)
tracker = CostTracker()

# Verify
print(f"Project root: {paths['project_root']}")
print(f"Data dir:     {paths['data']}")

# If you see a spaCy version warning later, update the model:
# !python -m spacy download en_core_web_lg

# --- INITIALIZE RUN ---
# This tracks all parameters and saves outputs to a unique folder
run = RunManager(run_name='ADS_Curation_Run')


Project root: c:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline
Data dir:     c:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline\data
ðŸš€ Run initialized: run_20260220_120219_ADS_Curation_Run
ðŸ“ All run outputs will be saved to: c:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline\runs\run_20260220_120219_ADS_Curation_Run


---
## Global Run Control

Set run-level switches here. Phase-specific parameters are configured in each phase section below.


In [2]:
# --- RUN CONTROL ---
# 0 = Run everything (Search to End)
# 4 = Load processed data (Skip Search, Translate, Tokenize) -> Start at Topic Modeling
START_AT_PHASE = 0
FORCE_REFRESH = False

# Shared OpenRouter pricing mode (used in translation, embeddings, topic labeling)
OPENROUTER_COST_MODE = "hybrid"  # 'hybrid', 'strict', 'fast'


---
# Phase 1: Search & Export

Query the NASA ADS API for publications matching a search string,
then resolve all bibcodes (publications + references) into full metadata.

### 1.1 Search Configuration


In [3]:
# --- SEARCH CONFIGURATION ---
# Modify the query string for your research question.
# See: https://ui.adsabs.harvard.edu/help/search/search-syntax

ADS_TOKEN = os.getenv("ADS_API_KEY")

Set_A = "docs(library/P0hyiLw0T8qsyHSBpWigJA)"
Set_B = f"citations({Set_A}) AND year:1915-2000"
Set_C = f"citations(citations({Set_A})) AND year:1915-2000"
Set_D = f"({Set_A}) OR ({Set_B}) OR ({Set_C})"
Set_T = "(docs(library/_-RCjJotRKCZC1PP03MqwA) AND year:1915-2000)"

gr_terms = [
    '(general AND relativi*)', 'gravit*', '(allgemein* AND relativi*)',
    '"relativitÃ© gÃ©nÃ©rale"', '"teoria della relativitÃ "',
    '"GravitÃ© quantique"', '"GravitÃ  quantistica"',
    '(einheitlich* AND feld*)', 'Quantengravitation', '"champ unifiÃ©"',
    '(unified AND field*)', '"quantum gravity"', 'cosmo*', 'Kosmo*',
]
Set_E = f"abs:({' OR '.join(gr_terms)}) AND year:1911-2000"

# Example quick focus query
QUERY = 'author:"Treder, H*"'
# Alternative broader query:
# QUERY = f"({Set_D}) OR ({Set_T}) OR ({Set_E})"


### 1.2 Execute Search

In [4]:
if START_AT_PHASE <= 1:
    from ads_bib.search import search_ads, save_search_results
    from ads_bib._utils.io import load_pickle
    
    latest = paths["raw"] / "search_results_latest.pkl"
    
    if FORCE_REFRESH or not latest.exists():
        bibcodes, references, esources, fulltext_urls = search_ads(QUERY, ADS_TOKEN)
        save_search_results((bibcodes, references, esources, fulltext_urls), paths["raw"])
    else:
        print(f"Loading cached results: {latest.name}")
        bibcodes, references, esources, fulltext_urls = load_pickle(latest)
    
    print(f"\nBibcodes:        {len(bibcodes):,}")
    print(f"Unique refs:     {len(set(r for rl in references for r in rl)):,}")
    print(f"Total refs:      {sum(len(rl) for rl in references):,}")
else:
    print(f"Skipping Phase 1 Search (START_AT_PHASE={START_AT_PHASE})")

Loading cached results: search_results_latest.pkl

Bibcodes:        382
Unique refs:     499
Total refs:      796


### 1.3 Export & Resolve Metadata

In [5]:
if START_AT_PHASE <= 1:
    from ads_bib.export import resolve_dataset
    from ads_bib._utils.io import save_json_lines, load_json_lines
    
    pubs_path = paths["raw"] / "publications.json"
    refs_path = paths["raw"] / "references.json"
    
    if FORCE_REFRESH or not pubs_path.exists():
        publications, refs = resolve_dataset(
            bibcodes, references, esources, fulltext_urls, ADS_TOKEN
        )
        save_json_lines(publications, pubs_path)
        save_json_lines(refs, refs_path)
    else:
        print("Loading cached exports ...")
        publications = load_json_lines(pubs_path)
        refs = load_json_lines(refs_path)
    
    print(f"\nPublications: {len(publications):,}")
    print(f"References:   {len(refs):,}")
    publications.info()

Loading cached exports ...

Publications: 382
References:   499
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 382 entries, 0 to 381
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Bibcode               382 non-null    object
 1   Author                382 non-null    object
 2   Title                 382 non-null    object
 3   Year                  382 non-null    int64 
 4   Journal               382 non-null    object
 5   Journal Abbreviation  382 non-null    object
 6   Issue                 382 non-null    object
 7   Volume                382 non-null    object
 8   First Page            382 non-null    object
 9   Last Page             382 non-null    object
 10  Abstract              382 non-null    object
 11  Keywords              382 non-null    object
 12  DOI                   382 non-null    object
 13  Affiliation           382 non-null    object
 14  Category              382 

In [6]:
if START_AT_PHASE <= 1:
    display(publications.head(10))

,Bibcode,Author,Title,Year,Journal,Journal Abbreviation,Issue,Volume,First Page,Last Page,Abstract,Keywords,DOI,Affiliation,Category,Citation Count,References,PDF_URL
0,1971grun.book.....T,"[Treder, H.J.]",Gravitationstheorie und Ã„quivalenzprinzip.,1971,Gravitationstheorie und Ã„quivalenzprinzip,grun.book,,,,,,,,AA(-),book,52,[],None
1,1957AnP...454..369T,"[Treder, H.]",Stromladungsdefinition und elektrische Kraft i...,1957,Annalen der Physik,AnP,6,454,369,380,"Wie Infeld 1950 gezeigt hat, ergibt sich aus d...",,10.1002/andp.19574540614,AA(-),article,34,[1953PhRv...92.1567C],None
2,1967AnP...475..194T,"[Treder, HansJÃ¼rgen]",Das makroskopische Gravitationsfeld in der ein...,1967,Annalen der Physik,AnP,3,475,194,206,A theory of the gravitation field is proposed ...,,10.1002/andp.19674750308,AA(-),article,38,"[1916AnP...354..769E, 1940PhRv...57..147R, 195...",None
3,1983AN....304..145T,"[Treder, H.J.]",The problem of the Grand Unification Theory,1983,Astronomische Nachrichten,AN,4,304,145,151,The evolution and fundamental questions of phy...,"Astronomical Models, Cosmology, Unified Field ...",10.1002/asna.2113040402,"AA(German Academy of Sciences, Potsdam)",article,1,[1950sts..book.....S],https://ui.adsabs.harvard.edu/link_gateway/198...
4,1972drdt.book.....T,"[Treder, H.J.]",Die RelativitÃ¤t der TrÃ¤gheit.,1972,Die RelativitÃ¤t der TrÃ¤gheit,drdt.book,,,,,,,,AA(-),book,31,[],None
5,1997GReGr..29..455V,"[von Borzeszkowski, HorstHeino, Treder, HansJÃ¼...",The Weyl-Cartan Space Problem in Purely Affine...,1997,General Relativity and Gravitation,GReGr,4,29,455,466,"According to PoincarÃ©, only the ``epistemologi...",,10.1023/A:1018830631884,AA(-); AB(-),article,21,"[1950sts..book.....S, 1971GReGr...2..313T, 199...",None
6,1975AnP...487..383T,"[Treder, H.J.]",Zur unitarisierten Gravitationstheorie mit lan...,1975,Annalen der Physik,AnP,,487,383,400,,,10.1002/andp.19754870509,AA(-),article,26,"[1963ForPh..11...81T, 1973AnP...485..229T, 197...",None
7,1981AN....302..275T,"[Treder, H. J., Jackisch, G.]",On Soldners Value of Newtonian Deflection of L...,1981,Astronomische Nachrichten,AN,6,302,275,,,,10.1002/asna.2103020603,AA(-); AB(-),article,6,"[1970AnP...480..315T, 1974AnP...486..325T]",https://ui.adsabs.harvard.edu/link_gateway/198...
8,1988mqg..book.....V,"[von Borzeszkowski, H.H., Treder, H.J.]",The meaning of quantum gravity.,1988,The meaning of quantum gravity.. H.-H. von Bor...,mqg..book,,20,,,Should the general theory of relativity be qua...,,,AA(-); AB(-),book,20,[],None
9,1980AnP...492..250T,"[Treder, H.J.]",Einstein's Hermitian theory of relativity as u...,1980,Annalen der Physik,AnP,,492,250,258,,"Relativistic Astrophysics, Gravitation Theory,...",10.1002/andp.19804920403,AA(-),article,19,"[1957AnP...454..369T, 1975AnP...487..375T]",None


---
# Phase 2: Translation

Detect non-English titles and abstracts with fasttext,
then translate them using either OpenRouter (API) or a local HuggingFace model.

### 2.1 Translation Configuration


In [7]:
# --- TRANSLATION CONFIGURATION ---
# Provider options: 'openrouter' (API) or 'huggingface' (local, with/without CUDA)

TRANSLATION_PROVIDER = "openrouter"  # or "huggingface"
TRANSLATION_MODEL = "google/gemini-3-flash-preview"
TRANSLATION_API_KEY = os.getenv("OPENROUTER_API_KEY")
TRANSLATION_MAX_WORKERS = 50

# Path to fasttext language ID model:
# https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin
FASTTEXT_MODEL = paths["models"] / "lid.176.bin"


### 2.2 Language Detection

In [8]:
if START_AT_PHASE <= 2:
    from ads_bib.translate import detect_languages
    
    print("=== Publications ===")
    publications = detect_languages(publications, ["Title", "Abstract"], model_path=FASTTEXT_MODEL)
    
    print("\n=== References ===")
    refs = detect_languages(refs, ["Title", "Abstract"], model_path=FASTTEXT_MODEL)
else:
    print(f"Skipping Phase 2 Translation (START_AT_PHASE={START_AT_PHASE})")

=== Publications ===
  Title: 183 non-English entries detected
  Abstract: 35 non-English entries detected

=== References ===
  Title: 185 non-English entries detected
  Abstract: 36 non-English entries detected


### 2.3 Translate Non-English Entries

In [9]:
if START_AT_PHASE <= 2:
    from ads_bib.translate import translate_dataframe
    
    print("=== Translating Publications ===")
    publications, cost_pubs = translate_dataframe(
        publications, ["Title", "Abstract"],
        provider=TRANSLATION_PROVIDER,
        model=TRANSLATION_MODEL,
        api_key=TRANSLATION_API_KEY,
        max_workers=TRANSLATION_MAX_WORKERS,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        cost_tracker=tracker,
    )
    
    print("\n=== Translating References ===")
    refs, cost_refs = translate_dataframe(
        refs, ["Title", "Abstract"],
        provider=TRANSLATION_PROVIDER,
        model=TRANSLATION_MODEL,
        api_key=TRANSLATION_API_KEY,
        max_workers=TRANSLATION_MAX_WORKERS,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        cost_tracker=tracker,
    )
    
    pub_cost = cost_pubs.get("cost_usd")
    ref_cost = cost_refs.get("cost_usd")
    print("\nTranslation Cost Snapshot:")
    print(f"  Publications: {'n/a' if pub_cost is None else f'${pub_cost:.4f}'}")
    print(f"  References:   {'n/a' if ref_cost is None else f'${ref_cost:.4f}'}")
    

=== Translating Publications ===
  Title: translating 183 entries with openrouter/google/gemini-3-flash-preview ...


  Title: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 183/183 [00:07<00:00, 24.77it/s]


  Abstract: translating 35 entries with openrouter/google/gemini-3-flash-preview ...


  Abstract: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 35/35 [00:03<00:00,  9.04it/s]


  Total translation cost: $0.0303 (218/218 priced; direct=218, fetched=0, mode=hybrid)

=== Translating References ===
  Title: translating 185 entries with openrouter/google/gemini-3-flash-preview ...


  Title: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 185/185 [00:17<00:00, 10.52it/s]


  Abstract: translating 36 entries with openrouter/google/gemini-3-flash-preview ...


  Abstract: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 36/36 [00:13<00:00,  2.68it/s]

  Total translation cost: $0.0317 (221/221 priced; direct=221, fetched=0, mode=hybrid)

Translation Cost Snapshot:
  Publications: $0.0303
  References:   $0.0317


---
# Phase 3: Tokenization

Create `full_text` (Title + Abstract) and tokenize with spaCy.

In [10]:
if START_AT_PHASE <= 3:
    from ads_bib.tokenize import tokenize_texts
    
    publications = tokenize_texts(publications)
    refs = tokenize_texts(refs)
else:
    print(f"Skipping Phase 3 Tokenization (START_AT_PHASE={START_AT_PHASE})")

Tokenizing 382 documents with en_core_web_lg ...
  Done.
Tokenizing 499 documents with en_core_web_lg ...
  Done.


### Checkpoint: Save Phase 1-3 Results

In [11]:
if START_AT_PHASE <= 3:
    from ads_bib._utils.io import save_json_lines
    import shutil

    # 1. Save to global cache so next time we can START_AT_PHASE = 4
    global_pub = paths["cache"] / "publications_translated_tokenized.json"
    global_ref = paths["cache"] / "references_translated_tokenized.json"
    
    save_json_lines(publications, global_pub)
    save_json_lines(refs, global_ref)
    
    # 2. Copy a static snapshot to the current Run folder
    shutil.copy(global_pub, run.paths["data"] / "publications_translated_tokenized.json")
    shutil.copy(global_ref, run.paths["data"] / "references_translated_tokenized.json")
    print("Checkpoint saved to global cache and local run folder.")


Checkpoint saved to global cache and local run folder.


---
# Phase 4: Author Name Disambiguation (Placeholder)

This step will use an external AND package once it's ready.
For now, the pipeline continues without disambiguation.

In [12]:
# === AND PLACEHOLDER ===
# Uncomment when the AND package is available:
#
# from and_package import disambiguate
# publications = disambiguate(publications)
# refs = disambiguate(refs)

print("AND step skipped (placeholder). Continuing without disambiguation.")

AND step skipped (placeholder). Continuing without disambiguation.


---
# Phase 5: Topic Modeling & Curation

Use BERTopic to discover thematic clusters, visualize with datamapplot,
then remove unwanted clusters to curate the dataset.

**Important:** Set parameters below based on your dataset size from Phase 1.

### 5.1 Embedding Configuration


In [13]:
# --- EMBEDDING CONFIGURATION ---
# Providers: 'local', 'huggingface_api', 'openrouter'

EMBEDDING_PROVIDER = "openrouter"
EMBEDDING_MODEL = "google/gemini-embedding-001"
EMBEDDING_API_KEY = os.getenv("OPENROUTER_API_KEY")

# For testing: set SAMPLE_SIZE to an integer (e.g. 200). None = full dataset.
SAMPLE_SIZE = None


### 5.2 Compute Embeddings

In [14]:
from ads_bib._utils.io import load_json_lines
import shutil

if START_AT_PHASE >= 4:
    # We skipped the earlier phases, so we must load the data from global cache
    global_pub = paths["cache"] / "publications_translated_tokenized.json"
    global_ref = paths["cache"] / "references_translated_tokenized.json"
    
    if not global_pub.exists():
        raise FileNotFoundError(f"Cannot skip to Phase 4. Global cache missing: {global_pub}")
        
    print("Loading data from global cache (Phase 1-3 skipped) ...")
    publications = load_json_lines(global_pub)
    refs = load_json_lines(global_ref)
    
    # Snapshot into current run for provenance
    shutil.copy(global_pub, run.paths["data"] / "publications_translated_tokenized.json")
    shutil.copy(global_ref, run.paths["data"] / "references_translated_tokenized.json")

print(f"Loaded {len(publications):,} publications")
print(f"Loaded {len(refs):,} references")


Loaded 382 publications
Loaded 499 references


In [15]:
from ads_bib.topic_model import compute_embeddings

df = publications.copy()
if SAMPLE_SIZE is not None:
    df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42).reset_index(drop=True)
    print(f"SAMPLING: {len(df):,} documents")

documents = df["full_text"].tolist()

embeddings = compute_embeddings(
    documents,
    provider=EMBEDDING_PROVIDER,
    model=EMBEDDING_MODEL,
    cache_dir=paths["embeddings_cache"],
    api_key=EMBEDDING_API_KEY,
    openrouter_cost_mode=OPENROUTER_COST_MODE,
    cost_tracker=tracker,
)
print(f"Embeddings shape: {embeddings.shape}")

  Loaded embeddings from cache: embeddings_openrouter_google_gemini-embedding-001.npz
Embeddings shape: (382, 3072)


### 5.3 Dimensionality Reduction Configuration

- Methods: `pacmap` (fast) or `umap` (preferred for hierarchical Toponymy structure).
- Heuristic: smaller/sparser sets use `n_neighbors` ~15; larger/denser sets often benefit from 50-60.
- Use `min_dist=0.0` for 5D clustering vectors and `min_dist=0.1` for 2D visualization.


In [16]:
# --- DIMENSIONALITY REDUCTION ---
DIM_REDUCTION_METHOD = "umap"  # 'pacmap' or 'umap'

PARAMS_5D = dict(n_neighbors=15, min_dist=0.0, metric="cosine", random_state=42)
PARAMS_2D = dict(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)

model_safe = EMBEDDING_MODEL.replace('/', '_')
CACHE_SUFFIX = f"{EMBEDDING_PROVIDER}_{model_safe}_{DIM_REDUCTION_METHOD}"


In [17]:
from ads_bib.topic_model import reduce_dimensions

reduced_5d, reduced_2d = reduce_dimensions(
    embeddings,
    method=DIM_REDUCTION_METHOD,
    params_5d=PARAMS_5D,
    params_2d=PARAMS_2D,
    cache_dir=paths["dim_reduction_cache"],
    cache_suffix=CACHE_SUFFIX,
)

  Loaded 5d_openrouter_google_gemini-embedding-001_umap from cache
  Loaded 2d_openrouter_google_gemini-embedding-001_umap from cache
  5D shape: (382, 5), 2D shape: (382, 2)


### 5.4 Clustering Configuration

**Adjust clustering granularity based on dataset size.**
- `MIN_CLUSTER_SIZE` controls BERTopic/HDBSCAN topic granularity (roughly ~0.1% of docs).
- `BASE_MIN_CLUSTER_SIZE` controls Toponymy micro-cluster granularity.
- `TOPONYMY_LAYER_INDEX` selects the fallback primary layer for visualization.


In [18]:
# --- CLUSTERING CONFIGURATION ---
CLUSTERING_METHOD = "fast_hdbscan"  # or 'hdbscan'

n_docs = len(df)
MIN_CLUSTER_SIZE = max(10, int(n_docs * 0.001))
BASE_MIN_CLUSTER_SIZE = max(10, int(n_docs * 0.001))
print(f"Dataset: {n_docs:,} documents")
print(f"  MIN_CLUSTER_SIZE={MIN_CLUSTER_SIZE}")
print(f"  BASE_MIN_CLUSTER_SIZE={BASE_MIN_CLUSTER_SIZE}")

CLUSTER_PARAMS = dict(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=3,
    cluster_selection_method="eom",
    cluster_selection_epsilon=0.02,
)

TOPONYMY_CLUSTER_PARAMS = dict(
    min_clusters=5,
    min_samples=3,
    base_min_cluster_size=BASE_MIN_CLUSTER_SIZE,
)

# EVoC uses a backend-specific constructor in some versions.
# Keep this dict conservative; unsupported keys are filtered in fit_toponymy.
TOPONYMY_EVOC_CLUSTER_PARAMS = dict(
    min_clusters=5,
    min_samples=3,
    base_min_cluster_size=BASE_MIN_CLUSTER_SIZE,
)

TOPONYMY_LAYER_INDEX = 0


Dataset: 382 documents
  MIN_CLUSTER_SIZE=10
  BASE_MIN_CLUSTER_SIZE=10


### 5.5 Backend & LLM Configuration

Backend matrix:
- `bertopic`: BERTopic on 5D reduced vectors + optional outlier reassignment refresh.
- `toponymy`: Toponymy + `ToponymyClusterer` on 5D reduced vectors.
- `toponymy_evoc`: Toponymy + `EVoCClusterer` on raw embeddings (no 5D clustering vectors).

`MIN_DF` guidance:
- small sample runs: often `1`
- full runs: usually `2+`


In [19]:
# --- TOPIC BACKEND + LLM LABELING ---
TOPIC_BACKEND = "toponymy_evoc"  # 'bertopic', 'toponymy', 'toponymy_evoc'

LLM_PROVIDER = "openrouter"
LLM_MODEL = "google/gemini-3-flash-preview"
LLM_API_KEY = os.getenv("OPENROUTER_API_KEY")

PIPELINE_MODELS = ["POS", "KeyBERT", "MMR"]
PARALLEL_MODELS = ["MMR", "POS", "KeyBERT"]

TOPONYMY_EMBEDDING_MODEL = EMBEDDING_MODEL
MIN_DF = 1 if (SAMPLE_SIZE and SAMPLE_SIZE < 1000) else 2


In [20]:
from ads_bib.topic_model import fit_bertopic, fit_toponymy, reduce_outliers, build_topic_dataframe
import numpy as np

selected_toponymy_cluster_params = (
    TOPONYMY_EVOC_CLUSTER_PARAMS
    if TOPIC_BACKEND == "toponymy_evoc"
    else TOPONYMY_CLUSTER_PARAMS
)

if TOPIC_BACKEND == "bertopic":
    topic_model = fit_bertopic(
        documents,
        reduced_5d,
        llm_provider=LLM_PROVIDER,
        llm_model=LLM_MODEL,
        pipeline_models=PIPELINE_MODELS,
        parallel_models=PARALLEL_MODELS,
        min_df=MIN_DF,
        clustering_method=CLUSTERING_METHOD,
        clustering_params=CLUSTER_PARAMS,
        api_key=LLM_API_KEY,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        cost_tracker=tracker,
    )

    topics = np.array(topic_model.topics_)
    topics = reduce_outliers(
        topic_model,
        documents,
        topics,
        reduced_5d,
        threshold=0.8,
        llm_provider=LLM_PROVIDER,
        llm_model=LLM_MODEL,
        api_key=LLM_API_KEY,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        cost_tracker=tracker,
    )
    topic_info = topic_model.get_topic_info()

elif TOPIC_BACKEND in {"toponymy", "toponymy_evoc"}:
    topic_model, topics, topic_info = fit_toponymy(
        documents,
        embeddings,
        reduced_5d,
        backend=TOPIC_BACKEND,
        layer_index=TOPONYMY_LAYER_INDEX,
        llm_provider=LLM_PROVIDER,
        llm_model=LLM_MODEL,
        embedding_model=TOPONYMY_EMBEDDING_MODEL,
        api_key=LLM_API_KEY,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        clusterer_params=selected_toponymy_cluster_params,
        cost_tracker=tracker,
    )

else:
    raise ValueError(
        f"Invalid TOPIC_BACKEND '{TOPIC_BACKEND}'. "
        "Use 'bertopic', 'toponymy', or 'toponymy_evoc'."
    )

df = build_topic_dataframe(
    df,
    topic_model,
    topics,
    reduced_2d,
    embeddings,
    topic_info=topic_info,
)
topic_info

  Using raw embeddings for clustering with EVoCClusterer.
  Clusterer: EVoCClusterer
Fitting Toponymy backend='toponymy_evoc' (LLM: openrouter/google/gemini-3-flash-preview) ...


Selecting facility_location exemplars:   0%|          | 0/12 [00:00<?, ?cluster/s]

embedding texts:   0%|          | 0/27 [00:00<?, ?it/s]

Building topic names by layer:   0%|          | 0/1 [00:00<?, ?layer/s]

Generating informative keyphrases:   0%|          | 0/12 [00:00<?, ?cluster/s]

Generating prompts for layer 0:   0%|          | 0/12 [00:00<?, ?topic/s]

Generating topic names for layer 0:   0%|          | 0/12 [00:00<?, ?topic/s]

  Selected Toponymy layer 0/0: 12 topics, 114 outliers.
  OpenRouter cost: $0.0056 (12/12 priced; direct=12, fetched=0, mode=hybrid)


,Topic,Name,Main
0,-1,Outlier Topic,Outlier Topic
1,0,"Fundamental Constants, Planck-Scale Physics, a...","Fundamental Constants, Planck-Scale Physics, a..."
2,1,Relativistic Generalizations and Covariant For...,Relativistic Generalizations and Covariant For...
3,2,"Historical Solar Variability, Solar-Terrestria...","Historical Solar Variability, Solar-Terrestria..."
4,3,Historical and Epistemological Intersections o...,Historical and Epistemological Intersections o...
5,4,"Mach's Principle, Riemannian Dynamics, and Pos...","Mach's Principle, Riemannian Dynamics, and Pos..."
6,5,Foundational Compatibility and Measurement Lim...,Foundational Compatibility and Measurement Lim...
7,6,"Mathematical Analysis of Singularities, Shock ...","Mathematical Analysis of Singularities, Shock ..."
8,7,"Unified Field Theory, Affine Gravity, and Herm...","Unified Field Theory, Affine Gravity, and Herm..."
9,8,Gravitational Effects on Light Propagation and...,Gravitational Effects on Light Propagation and...


### 5.6 Visualize Topics

In [21]:
from ads_bib.visualize import create_topic_map

# Use all layers for Toponymy, or just 'Name' for BERTopic
if TOPIC_BACKEND in {"toponymy", "toponymy_evoc"}:
    num_layers = len([col for col in df.columns if col.startswith("Topic_Layer_")])
    label_column = [f"Topic_Layer_{i}" for i in range(num_layers)]
else:
    label_column = "Name"

plot = create_topic_map(
    df,
    label_column=label_column,
    title="ADS Bibliometric Map",
    subtitle=f"Topics labeled with {LLM_PROVIDER}/{LLM_MODEL}",
    dark_mode=True,
    output_path=run.paths["plots"] / "topic_map.html",
)
# plot  # uncomment to display inline

Saved: topic_map.html


### 5.7 Curate Dataset

Review the cluster summary, then remove clusters that don't fit your research question.

In [22]:
from ads_bib.curate import get_cluster_summary, remove_clusters

get_cluster_summary(df)

,topic_id,Count,Percentage,Label
0,-1,114,29.8,Outlier Topic
8,7,50,13.1,"Unified Field Theory, Affine Gravity, and Herm..."
5,4,41,10.7,"Mach's Principle, Riemannian Dynamics, and Pos..."
11,10,40,10.5,Historical and Physical Foundations of Cosmogo...
1,0,28,7.3,"Fundamental Constants, Planck-Scale Physics, a..."
7,6,21,5.5,"Mathematical Analysis of Singularities, Shock ..."
2,1,20,5.2,Relativistic Generalizations and Covariant For...
9,8,15,3.9,Gravitational Effects on Light Propagation and...
4,3,11,2.9,Historical and Epistemological Intersections o...
6,5,11,2.9,Foundational Compatibility and Measurement Lim...


### Checkpoint: Save Curated Dataset

In [23]:
from ads_bib._utils.io import save_parquet

# Ensure References is list type for Parquet
df["References"] = df["References"].apply(lambda x: x if isinstance(x, list) else [])

save_parquet(df, run.paths["data"] / "curated_dataset.parquet")
print(f"Curated dataset: {len(df):,} documents")

Curated dataset: 382 documents


In [24]:
from ads_bib._utils.io import load_parquet

df = load_parquet(run.paths["data"] / "curated_dataset.parquet")
df.head(10)

,Bibcode,Author,Title,Year,Journal,Journal Abbreviation,Issue,Volume,First Page,Last Page,...,Abstract_en,full_text,tokens,embedding_2d_x,embedding_2d_y,topic_id,Name,Main,full_embeddings,Topic_Layer_0
0,1971grun.book.....T,"[Treder, H.J.]",Gravitationstheorie und Ã„quivalenzprinzip.,1971,Gravitationstheorie und Ã„quivalenzprinzip,grun.book,,,,,...,,Theory of Gravitation and the Equivalence Prin...,"[theory, gravitation, equivalence, principle]",12.903994,8.045989,9,Experimental Foundations of Gravitational Phys...,Experimental Foundations of Gravitational Phys...,"[0.00079854747, -0.019891707, 0.020806318, -0....",Experimental Foundations of Gravitational Phys...
1,1957AnP...454..369T,"[Treder, H.]",Stromladungsdefinition und elektrische Kraft i...,1957,Annalen der Physik,AnP,6,454,369,380,...,"As Infeld demonstrated in 1950, Einstein's str...",Definition of Electric Charge and Electric For...,"[definition, electric, charge, electric, force...",14.237543,6.162948,-1,Outlier Topic,Outlier Topic,"[-0.010657256, -0.003516388, 0.025264107, -0.0...",Unlabelled
2,1967AnP...475..194T,"[Treder, HansJÃ¼rgen]",Das makroskopische Gravitationsfeld in der ein...,1967,Annalen der Physik,AnP,3,475,194,206,...,A theory of the gravitation field is proposed ...,The Macroscopic Gravitational Field in the Uni...,"[macroscopic, gravitational, field, unified, q...",13.583593,5.302496,7,"Unified Field Theory, Affine Gravity, and Herm...","Unified Field Theory, Affine Gravity, and Herm...","[0.0009005609, -0.020405725, 0.010572005, -0.0...","Unified Field Theory, Affine Gravity, and Herm..."
3,1983AN....304..145T,"[Treder, H.J.]",The problem of the Grand Unification Theory,1983,Astronomische Nachrichten,AN,4,304,145,151,...,The evolution and fundamental questions of phy...,The problem of the Grand Unification Theory. T...,"[problem, grand, unification, theory, evolutio...",12.789717,5.674225,-1,Outlier Topic,Outlier Topic,"[-0.014908035, -0.018396216, 0.014744086, -0.0...",Unlabelled
4,1972drdt.book.....T,"[Treder, H.J.]",Die RelativitÃ¤t der TrÃ¤gheit.,1972,Die RelativitÃ¤t der TrÃ¤gheit,drdt.book,,,,,...,,The Relativity of Inertia..,"[relativity, inertia]",13.631322,8.390042,-1,Outlier Topic,Outlier Topic,"[0.022441277, -0.018025622, 0.012645888, -0.08...",Unlabelled
5,1997GReGr..29..455V,"[von Borzeszkowski, HorstHeino, Treder, HansJÃ¼...",The Weyl-Cartan Space Problem in Purely Affine...,1997,General Relativity and Gravitation,GReGr,4,29,455,466,...,"According to PoincarÃ©, only the ``epistemologi...",The Weyl-Cartan Space Problem in Purely Affine...,"[weyl, cartan, space, problem, purely, affine,...",13.896609,4.574677,7,"Unified Field Theory, Affine Gravity, and Herm...","Unified Field Theory, Affine Gravity, and Herm...","[-0.020331278, -0.015941117, 0.03069637, -0.06...","Unified Field Theory, Affine Gravity, and Herm..."
6,1975AnP...487..383T,"[Treder, H.J.]",Zur unitarisierten Gravitationstheorie mit lan...,1975,Annalen der Physik,AnP,,487,383,400,...,,On the unitarized theory of gravitation with l...,"[unitarize, theory, gravitation, short, range,...",13.163076,6.860907,-1,Outlier Topic,Outlier Topic,"[-0.0044219275, -0.009710963, 0.009115518, -0....",Unlabelled
7,1981AN....302..275T,"[Treder, H. J., Jackisch, G.]",On Soldners Value of Newtonian Deflection of L...,1981,Astronomische Nachrichten,AN,6,302,275,,...,,On Soldners Value of Newtonian Deflection of L...,"[soldners, value, newtonian, deflection, light]",12.283854,8.776442,8,Gravitational Effects on Light Propagation and...,Gravitational Effects on Light Propagation and...,"[0.014637167, -0.025634238, 0.013352524, -0.07...",Gravitational Effects on Light Propagation and...
8,1988mqg..book.....V,"[von Borzeszkowski, H.H., Treder, H.J.]",The meaning of quantum gravity.,1988,The meaning of quantum gravity.. H.-H. von Bor...,mqg..book,,20,,,...,Should the general theory of relativity be qua...,The meaning of quantum gravity.. Should the ge...,"[meaning, quantum, gravity, general, theor

---
# Phase 6: Citation Networks

Compute citation networks from the curated dataset and export
to GEXF (Gephi), Graphology JSON (Sigma.js), CSV, and/or Web of Science format.

### 6.1 Citation Configuration


In [25]:
# --- CITATION CONFIGURATION ---
CITE_METRICS = ["direct", "co_citation", "bibliographic_coupling", "author_co_citation"]
MIN_COUNTS = {
    "direct": 1,
    "co_citation": 1,
    "bibliographic_coupling": 1,
    "author_co_citation": 1,
}
AUTHORS_FILTER = None
OUTPUT_FORMAT = "gexf"  # 'gexf', 'graphology', 'csv', 'all'
CLUSTERS_TO_REMOVE = []

# Snapshot the full configuration once all phase configs are defined.
run.save_config(locals())


ðŸ’¾ Snapshot of configuration saved to config_used.yaml (38 parameters tracked).


### 6.2 Build Node Table & Compute Networks

In [26]:
from ads_bib.citations import build_all_nodes, process_all_citations, export_wos_format

# Reload processed data if needed
all_nodes = build_all_nodes(publications, refs)

results = process_all_citations(
    bibcodes, references, publications, refs, all_nodes,
    metrics=CITE_METRICS,
    min_counts=MIN_COUNTS,
    authors_filter=AUTHORS_FILTER,
    output_format=OUTPUT_FORMAT,
    output_dir=run.paths["data"],
)

All nodes: 773
Processing direct ...
  GEXF: direct.gexf
Processing co_citation ...
  GEXF: co_citation.gexf
Processing bibliographic_coupling ...
  GEXF: bibliographic_coupling.gexf
Processing author_co_citation ...
  GEXF: author_co_citation.gexf


### 6.3 Export Web of Science Format

In [27]:
suffix = "_filtered" if AUTHORS_FILTER else ""
export_wos_format(
    publications, refs,
    output_path=run.paths["data"] / f"download_wos_export{suffix}.txt",
)

  WOS format: download_wos_export.txt


---
# Summary

Final dataset statistics and output file listing.

In [28]:
import os

print("="*60)
print("PIPELINE COMPLETE")
print("="*60)
print(f"Publications:     {len(publications):,}")
print(f"References:       {len(refs):,}")
print(f"Curated dataset:  {len(df):,}")
print(f"Topics found:     {df['topic_id'].nunique()}")
print()
print("Output files:")
for root, dirs, files in os.walk(run.paths["root"]):
    for f in sorted(files):
        fpath = Path(root) / f
        size_mb = fpath.stat().st_size / 1024 / 1024
        print(f"  {fpath.relative_to(run.paths['root'])} ({size_mb:.1f} MB)")

print()
print(tracker.compact_summary())


PIPELINE COMPLETE
Publications:     382
References:       499
Curated dataset:  382
Topics found:     13

Output files:
  config_used.yaml (0.0 MB)
  data\author_co_citation.gexf (0.7 MB)
  data\bibliographic_coupling.gexf (0.8 MB)
  data\co_citation.gexf (2.1 MB)
  data\curated_dataset.parquet (5.4 MB)
  data\direct.gexf (1.6 MB)
  data\publications_translated_tokenized.json (0.7 MB)
  data\references_translated_tokenized.json (1.0 MB)
  plots\download_wos_export.txt (0.3 MB)
  plots\topic_map.html (0.2 MB)

CostTracker Summary (compact)
  translation | google/gemini-3-flash-preview | tokens(total=51,648, prompt=37,174, completion=14,474) | calls=2 | cost=$0.0620
  llm_labeling_toponymy_evoc | google/gemini-3-flash-preview | tokens(total=8,774, prompt=8,278, completion=496) | calls=1 | cost=$0.0056
  TOTAL | all models | tokens(total=60,422, prompt=-, completion=-) | calls=3 | cost=$0.0676
